In [1]:
!pip install statsmodels

In [2]:
import pandas as pd
import numpy as np
import zipfile
import matplotlib.pyplot as plt
import gc
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error

from tqdm import tqdm  # loading bar


import warnings
warnings.filterwarnings('ignore')

In [3]:
all_training_df = pd.read_csv("Cleaned Data/all_daily_training_data.csv")
forecast_pred = pd.read_csv("Cleaned Data/forecast_predictors.csv")
ss = pd.read_csv("SampleSubmission.csv")

In [4]:
all_training_df

,ID,kwh,Date,Source,v_label,consumer_device,data_user,Temperature (°C),Dewpoint Temperature (°C),U Wind Component (m/s),...,Total Precipitation (mm),Snowfall (mm),Snow Cover (%),year,month,day,day_of_week,week_of_year,quarter,is_weekend
0,2024-07-22_consumer_device_10_data_user_1,0.024330,2024-07-22,consumer_device_10_data_user_1,0,10,1,14.719596,8.280669,0.022655,...,0.073049,0.000000,0.000000,2024,7,22,0,30,3,0
1,2024-07-23_consumer_device_10_data_user_1,0.103560,2024-07-23,consumer_device_10_data_user_1,0,10,1,13.217268,9.862700,0.116137,...,0.121921,0.000000,0.000000,2024,7,23,1,30,3,0
2,2024-07-24_consumer_device_10_data_user_1,0.137543,2024-07-24,consumer_device_10_data_user_1,0,10,1,12.462190,9.865658,0.103451,...,0.119984,0.000000,0.000000,2024,7,24,2,30,3,0
3,2024-07-25_consumer_device_10_data_user_1,0.121011,2024-07-25,consumer_device_10_data_user_1,0,10,1,13.867551,8.973798,0.066345,...,0.034283,0.000000,0.000000,2024,7,25,3,30,3,0
4,2024-07-26_consumer_device_10_data_user_1,0.000000,2024-07-26,consumer_device_10_data_user_1,0,10,1,15.572609,9.434734,0.093025,...,0.006961,0.000000,0.000000,2024,7,26,4,30,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136404,2024-09-10_consumer_device_9_data_user_9,0.000000,2024-09-10,consumer_device_9_data_user_9,0,9,9,8.972100,4.735042,0.100475,...,0.012961,0.000000,0.000000,2024,9,10,1,37,3,0
136405,2024-09-11_consumer_device_9_data_user_9,0.000000,2024-09-11,consumer_device_9_data_user_9,0,9,9,9.216674,2.945390,0.031060,...,0.014010,0.000000,0.000000,2024,9,11,2,37,3,0
136406,2024-09-12_consumer_device_9_data_user_9,0.000000,2024-09-12,consumer_device_9_data_user_9,0,9,9,10.711587,1.898337,-0.069007,...,0.002685,0.000000,0.000000,2024,9,12,3,37,3,0
136407,2024-09-13_consumer_device_9_data_user_9,0.000000,2024-09-13,consumer_device_9_data_user_9,0,9,9,10.078417,3.285115,0.170258,...,0.009149,0.000000,0.000000,2024,9,13,4,37,3,0


In [5]:
all_training_df['Date'] = pd.to_datetime(all_training_df['Date'])

# Ensure complete datetime index
date_range = pd.date_range(
    start=all_training_df['Date'].min(),
    end=all_training_df['Date'].max()
)
df = all_training_df.set_index('Date')
# Sort the dataframe by date
df = df.sort_index()

# Calculate the split point 
# Create validation date range to be 30 days before the last date
split_date = df.index[-1] - pd.Timedelta(days=30) # index is inclusive exclusive

# Split the data
train_df = df.loc[:split_date].reset_index()
val_df = df.loc[split_date + pd.Timedelta(days=1):].reset_index()

test_df = forecast_pred

def calculate_rmse(actual, predicted):
    """
    Calculate the Root Mean Square Error (RMSE) between actual and predicted values.
    
    Parameters:
    actual (pd.Series or np.array): The actual observed values
    predicted (pd.Series or np.array): The predicted values
    
    Returns:
    float: The RMSE value
    """
    # Ensure both inputs are numpy arrays
    actual = np.array(actual)
    predicted = np.array(predicted)
    
    # Calculate the squared differences
    squared_diff = (actual - predicted) ** 2
    
    # Calculate the mean of squared differences
    mean_squared_diff = np.mean(squared_diff)
    
    # Calculate the square root of the mean squared differences
    rmse = np.sqrt(mean_squared_diff)
    
    return rmse

# Augmented Dickey-Fuller (ADF) 

The Augmented Dickey-Fuller (ADF) test is a statistical hypothesis test used to determine whether a unit root is present in a univariate time series dataset. In simpler terms, it helps us assess whether a time series is stationary or non-stationary.

**Purpose of the ADF Test**

The presence of a unit root in a time series indicates non-stationarity. Non-stationary data can exhibit trends, which can lead to inaccurate forecasts when using models that assume stationarity. The ADF test is employed to assess whether differencing the data (to achieve stationarity) is necessary before applying certain time series models.

**How the ADF Test Works**

1. Null Hypothesis : The null hypothesis of the ADF test is that the time series has a unit root, indicating it is non-stationary.
2. Alternative Hypothesis: The alternative hypothesis is that the time series is stationary (i.e., it does not have a unit root).
3. Test Statistic: The ADF test statistic is computed. This statistic is used to compare against critical values to determine the likelihood of rejecting the null hypothesis.
4. Critical Values: The ADF test provides critical values at various confidence levels. These critical values depend on the sample size and the chosen significance level.
5. Decision: Based on the test statistic and critical values, you can decide whether to reject or fail to reject the null hypothesis.

___
## Hyperparameter tunning (p, d, q) for each source specific

In [6]:
from statsmodels.tsa.stattools import acf, pacf
from scipy.linalg import LinAlgError

def perform_adf_test(data):
    # Perform ADF test
    result = adfuller(data)

    # Print the results
#     print('ADF Statistic:', result[0])
#     print('p-value:', result[1])
#     print('Critical Values:', result[4])

    # Check if the data is stationary based on the p-value
    if result[1] <= 0.05:
#         print("The data is stationary")
        return True
    else:
#         print("The data is not stationary, Data can be processed further")
        return False

def get_differencing_adf(data): 
    d = 0
    while (perform_adf_test(data) != True): # loop until the data is stationary
        d += 1
        data = data.diff().dropna()
    return d


``` python
import itertools
from statsmodels.tsa.arima.model import ARIMA

def find_best_arima(data, max_d, max_p=10, max_q=8):
    best_aic = float('inf')
    
    best_order = (0, 0, 0)
    
    # Test all combinations of p and q up to max_p/max_q
    for p, d, q in itertools.product(range(max_p+1), range(max_d+1), range(max_q+1)):
        try:
            model = ARIMA(data, order=(p, d, q)).fit()
            if model.aic < best_aic:
                best_aic = model.aic
                best_order = (p, d, q)
        except:
            continue
    
    return best_order, best_aic


# Initialize list to store results
results = []

# Wrap the groupby iterator with tqdm
total_groups = df['Source'].nunique()  # total number of sources
for source_name, group in tqdm(df.groupby('Source'), total=total_groups, desc="Processing Sources"):
    group_data = group['kwh'].dropna()
    d = get_differencing_adf(group_data)
    if len(group_data) < d:
        results.append({'Source': source_name, 'order': (0, 0, 0), 'AIC': 0 })
        continue
    best_order, best_aic = find_best_arima(group_data, d)
    results.append({'Source': source_name, 'order': best_order , 'AIC':best_aic })
    
order_df = pd.DataFrame(results)

order_df.to_csv("ARIMA_order_per_source.csv", index = False)
```
___
Processing Sources: 100%|██████████████████| 585/585 [15:06:42<00:00, 93.00s/it]

In [7]:
import ast
order_df = pd.read_csv("ARIMA_order_per_source.csv")

# Convert entire 'order' column using ast.literal_eval
order_df['order'] = order_df['order'].apply(
    lambda x: ast.literal_eval(x) if pd.notnull(x) else None
)

order_df

,Source,order,AIC
0,consumer_device_10_data_user_1,"(8, 1, 4)",370.857223
1,consumer_device_10_data_user_10,"(1, 0, 2)",412.148997
2,consumer_device_10_data_user_11,"(2, 2, 7)",355.233535
3,consumer_device_10_data_user_12,"(0, 0, 0)",233.468926
4,consumer_device_10_data_user_13,"(0, 0, 4)",167.865894
...,...,...,...
580,consumer_device_9_data_user_5,"(10, 0, 8)",246.116693
581,consumer_device_9_data_user_6,"(10, 0, 8)",548.360427
582,consumer_device_9_data_user_7,"(5, 0, 8)",387.247809
583,consumer_device_9_data_user_8,"(8, 0, 7)",255.117976


In [8]:
order_df['order'].value_counts()

order
(1, 0, 0)     37
(0, 0, 1)     15
(10, 0, 7)    15
(0, 1, 1)     14
(10, 0, 8)    13
              ..
(4, 1, 1)      1
(6, 0, 0)      1
(3, 1, 5)      1
(2, 1, 1)      1
(3, 1, 0)      1
Name: count, Length: 150, dtype: int64

In [9]:
def forecast_arima_order(input_data, forecast_horizon=30, output_template=None, source_order_df=None):
    # Convert Date column to datetime format
    input_data['Date'] = pd.to_datetime(input_data['Date'])

    # Extract consumer_device_x and data_user_y
    input_data[['consumer_device', 'data_user']] = input_data['Source'].str.extract(r'consumer_device_(\d+)_data_user_(\d+)')

    # Ensure data is sorted by consumer_device, data_user, and Date
    input_data = input_data.sort_values(by=['consumer_device', 'data_user', 'Date'])

    # Store forecasts
    forecast_results = []

    # Process each unique consumer_device_x and data_user_y combination
    for (consumer_device, data_user), group in input_data.groupby(["consumer_device", "data_user"]):
        # Set Date as index
        group = group.set_index("Date")
        
        # Get single source value using iloc[0]
        group_source = group['Source'].iloc[0]  # Get first occurrence
        source_order_row = source_order_df[source_order_df['Source'] == group_source]
        
        # Error handling for missing orders
        if source_order_row.empty:
            print(f"No order found for {group_source}")
            continue
            
        # Extract order tuple safely
        source_order = source_order_row['order'].iloc[0]

        # Ensure data is in the correct format
        group = group.asfreq('D').fillna(method='ffill')  # Fill missing dates with last known value

        # Fit ARIMA model
        try:
            model = ARIMA(group["kwh"], order=source_order)  # ARIMA(5,1,0) as a baseline  switch to
            fitted_model = model.fit()

            # Forecast for the next forecast_horizon days
            forecast_dates = pd.date_range(start=group.index[-1] + pd.Timedelta(days=1),
                                           periods=forecast_horizon, freq='D')
            forecast_values = fitted_model.forecast(steps=forecast_horizon)

            # Store results in required format
            forecast_df = pd.DataFrame({
                "ID": [f"{date.strftime('%Y-%m-%d')}_consumer_device_{consumer_device}_data_user_{data_user}"
                        for date in forecast_dates],
                "kwh": forecast_values
            })

            forecast_results.append(forecast_df)

        except Exception as e:
            print(f"Error processing {consumer_device}_{data_user}: {e}")

    # Combine all forecasts into a single DataFrame
    forecast_df = pd.concat(forecast_results, ignore_index=True)

    # If an output template is provided, align the output format
    if output_template is not None:
        output_template = output_template.drop(columns=['kwh'], errors='ignore')
        final_output = output_template.merge(forecast_df, on='ID', how='left').fillna(0)
    else:
        final_output = forecast_df

    return final_output

In [10]:
t = order_df[order_df['Source']=='consumer_device_9_data_user_5']
t['order'].iloc[0]

(10, 0, 8)

In [11]:
forecast = forecast_arima_order(input_data=train_df, forecast_horizon=30, output_template=val_df[['ID', 'kwh']], source_order_df= order_df)
rms_order = calculate_rmse(val_df['kwh'], forecast['kwh'])
rms_order

Error processing 35_6: LU decomposition error.


8.607291127455571

ARIMA model with a source specific hyperparemeters

Train RMSE = 8.607291127455571

Submission RMSE: 9.002148298


In [12]:
forecast_test = forecast

In [13]:
forecast = forecast_arima_order(input_data=all_training_df, forecast_horizon=30, output_template=ss, source_order_df=order_df)
forecast["kwh"] = forecast["kwh"].fillna(0)


In [14]:
forecast_ids = set(forecast['ID'])
ss_ids = set(ss['ID'])

# Find IDs present in forecast but not in ss
forecast_only_ids = forecast_ids - ss_ids

# Find IDs present in ss but not in forecast
ss_only_ids = ss_ids - forecast_ids

# Print the IDs that are in forecast but not in ss
print("IDs in 'forecast' but not in 'ss':")
print(forecast_only_ids)


# Print the IDs that are in ss but not in forecast
print("\nIDs in 'ss' but not in 'forecast':")
print(ss_only_ids)

# Print the number of IDs that differ
print(f"\nNumber of IDs that differ: {len(forecast_only_ids) + len(ss_only_ids)}")


IDs in 'forecast' but not in 'ss':
set()

IDs in 'ss' but not in 'forecast':
set()

Number of IDs that differ: 0



### ARIMA Model Order `(p, d, q)`:
Dataset is daily time-series data:
- **Autoregressive (`p=7`)**: The model uses patterns from the last 7 days of `kwh` values to predict future values.
- **Differencing (`d=1`)**: The model calculates daily differences (`kwh[t] - kwh[t-1]`) to remove trends and make the data stationary.  1-day difference
- **Moving Average (`q=3`)**: The model averages residuals (forecast errors) from the last 3 days to improve predictions. This step uses the past error to correct the prediction.
use the error from the prediction and actual to correct the next prediction



---
##### **Moving Average (`q`) in ARIMA How It Works**
1. The ARIMA model makes an initial prediction for a time step.
2. It calculates the **error** for that prediction:
   $$
   \text{Error} = \text{Actual Value} - \text{Predicted Value}
   $$
3. The model then incorporates these past errors (from up to 3 previous days when `q=3`) into its calculations to adjust future predictions.

This process helps account for patterns or randomness in the residuals that might not be captured by the autoregressive (`p`) or differencing (`d`) components.

- The moving average in ARIMA does **not** smooth actual values (like a rolling average). Instead, it adjusts predictions based on past forecast errors.
- A manually calculated moving average (e.g., using `rolling(window=3)`) smooths observed data by averaging actual values over a specified window.


In [15]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

# Function to process and forecast per unique consumer_device_x and data_user_y
def forecast_arima(input_data, forecast_horizon=30, output_template=None, p = 7, d = 1, q = 3):
    # Convert Date column to datetime format
    input_data['Date'] = pd.to_datetime(input_data['Date'])

    # Extract consumer_device_x and data_user_y
    input_data[['consumer_device', 'data_user']] = input_data['Source'].str.extract(r'consumer_device_(\d+)_data_user_(\d+)')

    # Ensure data is sorted by consumer_device, data_user, and Date
    input_data = input_data.sort_values(by=['consumer_device', 'data_user', 'Date'])

    # Store forecasts
    forecast_results = []

    # Process each unique consumer_device_x and data_user_y combination
    for (consumer_device, data_user), group in input_data.groupby(["consumer_device", "data_user"]):
        # Set Date as index
        group = group.set_index("Date")

        # Ensure data is in the correct format
        group = group.asfreq('D').fillna(method='ffill')  # Fill missing dates with last known value

        # Fit ARIMA model
        try:
            model = ARIMA(group["kwh"], order=(p, d, q))  # ARIMA(5,1,0) as a baseline  switch to
            fitted_model = model.fit()

            # Forecast for the next forecast_horizon days
            forecast_dates = pd.date_range(start=group.index[-1] + pd.Timedelta(days=1),
                                           periods=forecast_horizon, freq='D')
            forecast_values = fitted_model.forecast(steps=forecast_horizon)

            # Store results in required format
            forecast_df = pd.DataFrame({
                "ID": [f"{date.strftime('%Y-%m-%d')}_consumer_device_{consumer_device}_data_user_{data_user}"
                        for date in forecast_dates],
                "kwh": forecast_values
            })

            forecast_results.append(forecast_df)

        except Exception as e:
            print(f"Error processing {consumer_device}_{data_user}: {e}")

    # Combine all forecasts into a single DataFrame
    forecast_df = pd.concat(forecast_results, ignore_index=True)

    # If an output template is provided, align the output format
    if output_template is not None:
        output_template = output_template.drop(columns=['kwh'], errors='ignore')
        final_output = output_template.merge(forecast_df, on='ID', how='left').fillna(0)
    else:
        final_output = forecast_df

    return final_output

### Test different model hyper parameters

In [16]:
forecast = forecast_arima(input_data=train_df, forecast_horizon=30, output_template=val_df[['ID', 'kwh']], p = 5, q = 1, d = 0)
rms510 = calculate_rmse(val_df['kwh'], forecast['kwh'])
rms510

8.720043497960631

In [17]:
forecast = forecast_arima(input_data=train_df, forecast_horizon=30, output_template=val_df[['ID', 'kwh']], p = 6, d = 1, q = 3)
rms613 = calculate_rmse(val_df['kwh'], forecast['kwh'])
rms613

8.521947696123739

In [18]:
forecast_702 = forecast_arima(input_data=train_df, forecast_horizon=30, output_template=val_df[['ID', 'kwh']], p = 7, d = 0, q = 2)
rms702 = calculate_rmse(val_df['kwh'], forecast_702['kwh'])
rms702

8.73175993014636

In [19]:
forecast = forecast_arima(input_data=train_df, forecast_horizon=30, output_template=val_df[['ID', 'kwh']], p = 7, d = 1, q = 3)
calculate_rmse(val_df['kwh'], forecast['kwh'])
rms713 = calculate_rmse(val_df['kwh'], forecast['kwh'])
rms713

Error processing 2_3: LU decomposition error.


8.809956482182832

In [20]:
forecast = forecast_arima(input_data=train_df, forecast_horizon=30, output_template=val_df[['ID', 'kwh']], p = 7, d = 1, q = 4)
calculate_rmse(val_df['kwh'], forecast['kwh'])
rms714 = calculate_rmse(val_df['kwh'], forecast['kwh'])
rms714

8.673224757610807

In [21]:
forecast = forecast_arima(input_data=all_training_df, forecast_horizon=30, output_template=ss, p = 7, d = 1, q = 3)

Check if the model is the same same as last run

In [22]:
submitted_forecast = pd.read_csv("forecast_ARIMA__7_1_3_.csv")
result = np.isclose(submitted_forecast['kwh'] , forecast['kwh'], rtol=1e-5, atol=1e-8)
all_equal = result.all()
all_equal

True

In [36]:
forecast = forecast_arima(input_data=all_training_df, forecast_horizon=30, output_template=ss , p = 7, d = 1, q = 4)


In [37]:
forecast.head()

,ID,kwh
0,2024-09-24_consumer_device_12_data_user_1,0.163310
1,2024-09-25_consumer_device_12_data_user_1,0.144851
2,2024-09-26_consumer_device_12_data_user_1,0.182648
3,2024-09-27_consumer_device_12_data_user_1,0.161827
4,2024-09-28_consumer_device_12_data_user_1,0.136037


In [38]:
# prompt: does forecast["kwh"] contain nans if so replace with 0

# Check for NaN values in the 'kwh' column and replace them with 0
forecast["kwh"] = forecast["kwh"].fillna(0)


In [39]:
len(all_training_df), len(forecast), len(ss)

(136409, 6014, 6014)

In [40]:
forecast.to_csv("forecast_ARIMA_(7,1,4).csv", index = False)

In [41]:
# prompt: list the difference in the ID between forecast and ss

# Assuming 'forecast' and 'ss' DataFrames are already defined as in your provided code.

# Convert 'ID' columns to sets for efficient comparison
forecast_ids = set(forecast['ID'])
ss_ids = set(ss['ID'])

# Find IDs present in forecast but not in ss
forecast_only_ids = forecast_ids - ss_ids

# Find IDs present in ss but not in forecast
ss_only_ids = ss_ids - forecast_ids

# Print the IDs that are in forecast but not in ss
print("IDs in 'forecast' but not in 'ss':")
print(forecast_only_ids)


# Print the IDs that are in ss but not in forecast
print("\nIDs in 'ss' but not in 'forecast':")
print(ss_only_ids)

# Print the number of IDs that differ
print(f"\nNumber of IDs that differ: {len(forecast_only_ids) + len(ss_only_ids)}")


IDs in 'forecast' but not in 'ss':
set()

IDs in 'ss' but not in 'forecast':
set()

Number of IDs that differ: 0


In [42]:
# prompt: compute RMSE score between forecast and ss

import pandas as pd
from sklearn.metrics import mean_squared_error
import math

# Assuming 'forecast' and 'ss' are DataFrames with a common 'ID' column and a 'kwh' column
# containing the forecast and actual values respectively.

# Merge the forecast and ss DataFrames on the 'ID' column
merged_df = pd.merge(forecast, ss, on='ID', how='left', suffixes=('_forecast', '_actual'))

# Calculate the RMSE
rmse = math.sqrt(mean_squared_error(merged_df['kwh_actual'], merged_df['kwh_forecast']))

print(f"RMSE: {rmse}")


RMSE: 12.81846649955261


### ARIMA Evaluation
Train RMSE is compare to 0

ARIMA (5,1,0)

Train RMSE: 8.720043497960631

Submission RMSE: 8.38863658

ARIMA (6, 1, 3)
Order (6, 1, 3): Average AIC across all groups: -482.99233798858523

Train RMSE: 8.720043497960631

Submission RMSE: 8.135190388


ARIMA (7, 0, 2)
Order (7, 0, 2): Average AIC across all groups: -483.5011645121932

Train RMSE: 8.73175993014636

Submission RMSE: 7.079242261


**ARIMA (7, 1, 3)**
Order (7, 1, 3): Average AIC across all groups: -482.26954080898076

Train RMSE: 8.809956482182832

**Submission RMSE: 6.987309958**



Order (7, 1, 4): Average AIC across all groups: -482.9539904169344

Train RMSE: 8.673224757610807

Submission RMSE: 8.860194086


### Gridsearch for hyperparameter tuning

best order for the best average hyperparameter aic score for all of the sources

```
Order (0, 0, 0): Average AIC across all groups: -203.83576186217368
Order (0, 0, 1): Average AIC across all groups: -330.35132195895335
Order (0, 0, 2): Average AIC across all groups: -382.7558905891822
Order (0, 0, 3): Average AIC across all groups: -415.72350718471733
Order (0, 0, 4): Average AIC across all groups: -427.9364420717181
Order (0, 1, 0): Average AIC across all groups: -414.7939675705068
Order (0, 1, 1): Average AIC across all groups: -452.40182660433413
Order (0, 1, 2): Average AIC across all groups: -461.8393346046657
Order (0, 1, 3): Average AIC across all groups: -466.4568192097755
Order (0, 1, 4): Average AIC across all groups: -470.7874658196664
Order (1, 0, 0): Average AIC across all groups: -441.76838960146466
Order (1, 0, 1): Average AIC across all groups: -460.3000936763226
Order (1, 0, 2): Average AIC across all groups: -464.9116451152533
Order (1, 0, 3): Average AIC across all groups: -466.2196736311925
Order (1, 0, 4): Average AIC across all groups: -467.0003225541373
Order (1, 1, 0): Average AIC across all groups: -439.04568676514793
Order (1, 1, 1): Average AIC across all groups: -459.9461161059043
Order (1, 1, 2): Average AIC across all groups: -464.22113783979086
Order (1, 1, 3): Average AIC across all groups: -467.6413226495705
Order (1, 1, 4): Average AIC across all groups: -471.37925558684236
Order (2, 0, 0): Average AIC across all groups: -455.30455293000693
Order (2, 0, 1): Average AIC across all groups: -459.94079161363527
Order (2, 0, 2): Average AIC across all groups: -462.41408414817175
Order (2, 0, 3): Average AIC across all groups: -465.6167385764107
Order (2, 0, 4): Average AIC across all groups: -470.56522991065043
Order (2, 1, 0): Average AIC across all groups: -449.61486125861245
Order (2, 1, 1): Average AIC across all groups: -464.3877804038549
Order (2, 1, 2): Average AIC across all groups: -468.9559321726169
Order (2, 1, 3): Average AIC across all groups: -473.79356718556465
Order (2, 1, 4): Average AIC across all groups: -476.6413030654537
Order (3, 0, 0): Average AIC across all groups: -460.0373944914775
Order (3, 0, 1): Average AIC across all groups: -465.10729550760396
Order (3, 0, 2): Average AIC across all groups: -467.5571429485227
Order (3, 0, 3): Average AIC across all groups: -471.1127909679567
Order (3, 0, 4): Average AIC across all groups: -474.30207752328045
Order (3, 1, 0): Average AIC across all groups: -455.38209999155066
Order (3, 1, 1): Average AIC across all groups: -466.3713905734657
Order (3, 1, 2): Average AIC across all groups: -471.67821654128716
Order (3, 1, 3): Average AIC across all groups: -474.7639000130278
Order (3, 1, 4): Average AIC across all groups: -480.159051386056
Order (4, 0, 0): Average AIC across all groups: -463.7035951463672
Order (4, 0, 1): Average AIC across all groups: -466.2285096845503
Order (4, 0, 2): Average AIC across all groups: -468.0337274897948
Order (4, 0, 3): Average AIC across all groups: -469.9043609990187
Order (4, 0, 4): Average AIC across all groups: -475.2669350076967
Order (4, 1, 0): Average AIC across all groups: -463.4230176893618
Order (4, 1, 1): Average AIC across all groups: -471.5740837935817
Order (4, 1, 2): Average AIC across all groups: -474.0936917999649
Order (4, 1, 3): Average AIC across all groups: -478.81721863962474
Order (4, 1, 4): Average AIC across all groups: -480.12119307950184
Order (5, 0, 0): Average AIC across all groups: -468.3649128708058
Order (5, 0, 1): Average AIC across all groups: -469.83065119540015
Order (5, 0, 2): Average AIC across all groups: -469.0267051035023
Order (5, 0, 3): Average AIC across all groups: -473.01280414771156
Order (5, 0, 4): Average AIC across all groups: -476.86374528224593
Order (5, 1, 0): Average AIC across all groups: -468.09210823690466
Order (5, 1, 1): Average AIC across all groups: -474.0128125041643
Order (5, 1, 2): Average AIC across all groups: -478.92644801569605
Order (5, 1, 3): Average AIC across all groups: -477.7837993456604
Order (5, 1, 4): Average AIC across all groups: -479.8236642331646
Order (6, 0, 0): Average AIC across all groups: -471.7669598328318
Order (6, 0, 1): Average AIC across all groups: -472.55525640172954
Order (6, 0, 2): Average AIC across all groups: -475.4243047858249
Order (6, 0, 3): Average AIC across all groups: -475.7513573647801
Order (6, 0, 4): Average AIC across all groups: -476.6098705269013
Order (6, 1, 0): Average AIC across all groups: -474.1909469695566
Order (6, 1, 1): Average AIC across all groups: -477.17861995280214
Order (6, 1, 2): Average AIC across all groups: -480.81130628138055
Order (6, 1, 3): Average AIC across all groups: -482.99233798858523
Order (6, 1, 4): Average AIC across all groups: -482.28387000628334
Order (7, 0, 0): Average AIC across all groups: -476.04285095082196
Order (7, 0, 1): Average AIC across all groups: -477.20420470873836
Order (7, 0, 2): Average AIC across all groups: -483.5011645121932
Order (7, 0, 3): Average AIC across all groups: -478.93785438532217
Order (7, 0, 4): Average AIC across all groups: -480.32605821755766
Order (7, 1, 0): Average AIC across all groups: -477.2666253958193
Order (7, 1, 1): Average AIC across all groups: -479.2207671855526
Order (7, 1, 2): Average AIC across all groups: -481.08450974343606
Order (7, 1, 3): Average AIC across all groups: -482.26954080898076
Order (7, 1, 4): Average AIC across all groups: -482.9539904169344
Global best order: (7, 0, 2), Average AIC: -483.5011645121932
Best ARIMA order globally: (7, 0, 2)
```